In [1]:
import pandas as pd

orders = pd.read_csv("olist_orders_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")

In [2]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [3]:
for name, df in {
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "customers": customers,
    "products": products
}.items():
    print(name, df.shape)

orders (99441, 8)
order_items (112650, 7)
payments (103886, 5)
customers (99441, 5)
products (32951, 9)


In [7]:
master_df = orders.copy()

master_df = master_df.merge(order_items, on="order_id", how="left")
#print(master_df.shape)

master_df = master_df.merge(payments, on="order_id", how="left")
#print(master_df.shape)

master_df = master_df.merge(customers, on="customer_id", how="left")
#print(master_df.shape)

master_df = master_df.merge(products, on="product_id", how="left")

#final shape of the data
print(master_df.shape)

(118434, 30)


In [8]:
#missing values
print(master_df.isnull().sum().sort_values(ascending=False).head(10))

#duplicates
print(master_df.duplicated().sum())

#sample
master_df.head()

order_delivered_customer_date    3397
product_photos_qty               2528
product_category_name            2528
product_name_lenght              2528
product_description_lenght       2528
order_delivered_carrier_date     2074
product_weight_g                  850
product_length_cm                 850
product_height_cm                 850
product_width_cm                  850
dtype: int64
0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,customer_city,customer_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,...,sao paulo,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,...,sao paulo,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,...,sao paulo,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,...,barreiras,BA,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,...,vianopolis,GO,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0


In [9]:
#Creating 10% duplicates
dup_sample = master_df.sample(frac=0.1, random_state=42)
master_df = pd.concat([master_df, dup_sample], ignore_index=True)

print("After adding duplicates: ", master_df.shape)

After adding duplicates:  (130277, 30)


In [10]:
#Customer city inconsistency

master_df.loc[master_df.sample(frac=0.2).index, "customer_city"]
master_df.loc[master_df.sample(frac=0.2).index, "customer_city"]

29193                itumbiara
38481                  goiania
63090     sao jose dos pinhais
125368               americana
40286                 campinas
                  ...         
40881                guarapari
6859                  curitiba
39676           rio de janeiro
13033             porto alegre
34233           ribeirao preto
Name: customer_city, Length: 26055, dtype: object

In [12]:
# messing dates randomly
sample_idx = master_df.sample(frac=0.2, random_state=42).index

master_df.loc[sample_idx, "order_purchase_timestamp"] = pd.to_datetime(
    master_df.loc[sample_idx, "order_purchase_timestamp"]
).dt.strftime("%d/%m/%Y")

sample_idx2 = master_df.sample(frac=0.2, random_state=1).index

master_df.loc[sample_idx2, "order_purchase_timestamp"] = pd.to_datetime(
    master_df.loc[sample_idx2, "order_purchase_timestamp"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
).dt.strftime("%b %d %Y")

C:\Users\Harsh\AppData\Local\Temp\ipykernel_16120\35561050.py:4: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  master_df.loc[sample_idx, "order_purchase_timestamp"] = pd.to_datetime(


In [13]:
sample_idx = master_df.sample(frac=0.1, random_state=42).index

master_df.loc[sample_idx, 'price'] = master_df.loc[sample_idx, 'price'].astype(str) + " USD"

C:\Users\Harsh\AppData\Local\Temp\ipykernel_16120\121811935.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['34.9 USD' '60.5 USD' '39.0 USD' ... '153.0 USD' '233.91 USD' '119.0 USD']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  master_df.loc[sample_idx, 'price'] = master_df.loc[sample_idx, 'price'].astype(str) + " USD"


In [15]:
# converting price safely to numeric
price_numeric = master_df['price'].astype(str).str.extract(r'(\d+\.?\d*)')[0]
price_numeric = price_numeric.astype(float)

# now adding negative values
neg_idx = master_df.sample(frac=0.02, random_state=42).index
master_df.loc[neg_idx, 'price'] = -abs(price_numeric.loc[neg_idx])

In [16]:
master_df.to_csv("raw_dirty_ecommerce_data.csv", index=False)